[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tralev/murmuration/blob/main/notebooks/murmuration.ipynb)

# Murmuration — Emergent Marginal Opacity

This notebook demonstrates the **hybrid projection model** from Pearce et al. (2014, PNAS).
We run a headless simulation and track the three key scientific metrics:

- **Θ** (internal opacity) — average fraction of each bird's view occluded by other birds
- **Θ′** (external opacity) — fraction of sky obscured from a distant observer
- **α** (order parameter) — degree of flock alignment (0 = chaotic, 1 = perfectly aligned)

The paper predicts that flocks self-organise to **marginal opacity** (Θ ≈ 0.25–0.60) —
neither fully transparent nor fully opaque — as an emergent property of the projection model.

## 1. Setup — Import Modules & Configure Headless Mode

In [ ]:
import os, sys

# Ensure project root is on path (works from any directory, including nbconvert)
_cwd = os.path.abspath(os.getcwd())
_root = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'notebooks' else _cwd
if os.path.exists(os.path.join(_root, 'flock_core.py')):
    sys.path.insert(0, _root)

# Headless mode — no display required
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['SDL_AUDIODRIVER'] = 'dummy'

import pygame
pygame.init()

# Project modules
from flock_core import (
    WIDTH, HEIGHT, FPS, NUM_BOIDS, V0, BOID_SIZE,
    LOG_EVERY, MODE_PROJECTION, Config, SpatialGrid, VISUAL_RANGE,
)
from extensions.predator import PredatorBoid
from extensions.spatial_optimization import SpatialChunker
from extensions.multi_viewpoint_opacity import FlockMetricsExtended

print(f'Pygame {pygame.version.ver} — headless mode ready')
print(f'Simulation area: {WIDTH}x{HEIGHT}, {NUM_BOIDS} birds, V0={V0}')

## 2. Run Headless Simulation — Collect Metrics

We run for ~600 frames (10 seconds at 60 FPS) and record Θ, Θ′, α, and FPS every 5 frames.

In [ ]:
from IPython.display import clear_output

# Configuration
config = Config()
config.mode = MODE_PROJECTION
N_FRAMES = 600
SAMPLE_EVERY = 5

# Initialise flock, metrics, and spatial chunker
flock = [PredatorBoid() for _ in range(config.num_boids)]
metrics = FlockMetricsExtended()
chunker = SpatialChunker()
clock = pygame.time.Clock()

# Data collection
records = []

print(f'Running {N_FRAMES} frames with {len(flock)} birds...')

for frame in range(N_FRAMES):
    dt = clock.tick(FPS)
    
    chunker.rebuild(flock)
    for boid in flock:
        boid._chunker = chunker
    for boid in flock:
        boid.flock(flock, config)
    for boid in flock:
        boid.update()
    metrics.update(flock, clock, config)
    
    if frame % SAMPLE_EVERY == 0:
        records.append((frame, metrics.internal_opacity,
                        metrics.external_opacity, metrics.order_param,
                        clock.get_fps()))
    
    if frame % 100 == 0:
        clear_output(wait=True)
        print(f'Frame {frame}/{N_FRAMES}  |  Theta={metrics.internal_opacity:.3f}  '
              f'FPS={clock.get_fps():.0f}')

clear_output(wait=True)
print(f'Done. {len(records)} samples over {N_FRAMES} frames.')
print(f'Final: Theta={metrics.internal_opacity:.3f}  alpha={metrics.order_param:.3f}')

## 3. Plot — Emergent Marginal Opacity

The shaded band shows the **marginal opacity region** (Θ ≈ 0.25–0.60) observed in real starling
flocks by Pearce et al. (2014).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

frames  = [r[0] for r in records]
theta   = [r[1] for r in records]
theta_x = [r[2] for r in records]
alpha   = [r[3] for r in records]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.axhspan(0.25, 0.60, alpha=0.12, color='green', label='Marginal opacity band')
ax1.plot(frames, theta,   linewidth=1.2, color='#3b82f6', label='Theta (internal)')
ax1.plot(frames, theta_x, linewidth=1.2, color='#f59e0b', label="Theta' (external)")
ax1.set_ylabel('Opacity', fontsize=12)
ax1.set_ylim(0, 1.05)
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_title('Emergent Opacity — Hybrid Projection Model', fontsize=14, fontweight='bold')

ax2.plot(frames, alpha, linewidth=1.2, color='#8b5cf6', label='alpha (order parameter)')
ax2.axhline(1.0, linestyle='--', linewidth=0.8, color='gray', alpha=0.5)
ax2.set_xlabel('Frame', fontsize=12)
ax2.set_ylabel('Order Parameter alpha', fontsize=12)
ax2.set_ylim(0, 1.05)
ax2.legend(loc='upper right', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_title('Flock Alignment', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

steady = theta[-200:]
print(f'Steady-state (last 200): Theta mean = {np.mean(steady):.4f} +/- {np.std(steady):.4f}')

## 4. Summary

This notebook demonstrates the three key predictions of the Pearce et al. (2014) model:

1. **Marginal Opacity**: Theta converges into the 0.25–0.60 band without explicit tuning.
2. **Order Parameter**: alpha stabilises near 0.8–0.95, indicating strong flock alignment.
3. **Emergent Behaviour**: No explicit cohesion or separation forces — the projection term
   delta_hat alone keeps the flock together.

---

*Pearce, D. J. G., Miller, A. M., Rowlands, G., & Turner, M. S. (2014). Role of projection in the control of bird flocks. PNAS, 111(29), 10422–10426.*